In [1]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from datasets import Dataset

/mnt/c/Users/Anurag Lawaniya/Desktop/tf-gpu-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# UPDATE THIS IN CELL 3
df = pd.read_csv("final_hard_negative_dataset.csv")



In [3]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


In [4]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    # This correctly populates 'token_type_ids', telling BERT where Resume ends and JD begins
    return tokenizer(
        example["resume_text"], 
        example["jd_text"],
        truncation=True,
        padding="max_length",
        max_length=512  # Increased from 256 to fit both documents
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|█████████████████████████████████████████████████████████████████| 2309/2309 [00:00<00:00, 4107.78 examples/s]


In [5]:
train_dataset = train_dataset.rename_column("final_score", "labels")
val_dataset = val_dataset.rename_column("final_score", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [6]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=1  # regression
)


Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2806.72it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initiali

In [7]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()

    mse = mean_squared_error(labels, preds)
    mae = mean_absolute_error(labels, preds)

    return {
        "mse": mse,
        "mae": mae
    }

In [8]:
training_args = TrainingArguments(
    output_dir="./results_8",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-6,
    save_strategy="no",
   
)


In [9]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Is CUDA available? True
Device Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.016548,0.004537,0.004537,0.052721
2,0.004935,0.005771,0.005771,0.063806
3,0.003375,0.003743,0.003743,0.049363
4,0.002741,0.002400,0.002400,0.039192
5,0.002258,0.002941,0.002941,0.043964
6,0.001957,0.002441,0.002441,0.039756
7,0.001702,0.002273,0.002273,0.038514
8,0.001497,0.002042,0.002042,0.036076
9,0.001347,0.001737,0.001737,0.033176
10,0.001230,0.001875,0.001875,0.034710


TrainOutput(global_step=46180, training_loss=0.003759130836823845, metrics={'train_runtime': 12671.7784, 'train_samples_per_second': 7.288, 'train_steps_per_second': 3.644, 'total_flos': 2.42980877978112e+16, 'train_loss': 0.003759130836823845, 'epoch': 10.0})

In [12]:
# Save model + tokenizer
trainer.save_model("bert_resume_match_model_8")
tokenizer.save_pretrained("bert_resume_match_model_8")

Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.89s/it]


('bert_resume_match_model_8/tokenizer_config.json',
 'bert_resume_match_model_8/tokenizer.json')

In [13]:
# UPDATE THIS IN CELL 14
def predict_score(resume, jd):
     
    inputs = tokenizer(
    resume, 
    jd,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    score = outputs.logits.squeeze().item()
    score = max(0, min(1, score))

    return score

In [14]:
print(predict_score("Python ML Flask", "Looking for ML engineer"))

0.5925722718238831


In [15]:
print(predict_score("java springboot engineer", "Looking for ML engineer"))

0.4253476858139038


In [16]:
# The Job Description
ml_jd = """
Looking for a Machine Learning Engineer to build and deploy deep learning models. 
The ideal candidate is proficient in Python and frameworks like PyTorch or TensorFlow. 
Experience with NLP and Scikit-learn is required for this role.
"""

# Profile A: Perfect Match (Python/ML)
python_resume = """
Senior Data Scientist with 4 years of experience in Python and Machine Learning. 
Expert in building NLP pipelines using PyTorch and Scikit-learn. 
Developed several deep learning models for sentiment analysis and predictive modeling.
"""

# Profile B: Hard Negative (Great Developer, Wrong Stack)
java_resume = """
Senior Backend Developer with 10 years of experience in Java and Spring Boot. 
Expert in building scalable microservices and managing Oracle SQL databases. 
Proven track record of delivering enterprise-level web applications and APIs.
"""

# Testing the model
print(f"Python Match Score: {predict_score(python_resume, ml_jd):.4f}")
print(f"Java Match Score: {predict_score(java_resume, ml_jd):.4f}")

Python Match Score: 0.8051
Java Match Score: 0.2608


In [17]:
jd_ds = "Looking for a Data Scientist to build predictive models using Python, XGBoost, and Scikit-learn. Must have experience with model deployment."

# Match (Data Scientist)
res_ds_match = "Data Scientist with expertise in predictive modeling, feature engineering, and Python. Proficient in XGBoost and deploying models via Flask."

# Partial Match (Data Analyst - knows Python but not ML)
res_ds_partial = "Data Analyst with 5 years of experience in SQL, Tableau, and Python for data cleaning. Focused on business intelligence and reporting."

print(f"DS Match: {predict_score(res_ds_match, jd_ds):.4f}")
print(f"DS Partial: {predict_score(res_ds_partial, jd_ds):.4f}")

DS Match: 0.8418
DS Partial: 0.2483


In [18]:
jd_frontend = "Senior Frontend Engineer specialized in React.js, TypeScript, and Redux. Experience with responsive design and Tailwind CSS is required."

# Match (Frontend)
res_fe_match = "Senior React Developer with 6 years of experience building complex SPAs using TypeScript and Redux. Expert in CSS-in-JS and Tailwind."

# Hard Negative (Backend Node.js - same language, wrong side of the stack)
res_fe_hard_neg = "Senior Backend Engineer specialized in Node.js, Express, and PostgreSQL. Expert in building RESTful APIs and microservices architecture."

print(f"Frontend Match: {predict_score(res_fe_match, jd_frontend):.4f}")
print(f"Frontend Hard Negative: {predict_score(res_fe_hard_neg, jd_frontend):.4f}")

Frontend Match: 0.8257
Frontend Hard Negative: 0.4389


In [19]:
jd_devops = "DevOps Engineer to manage AWS infrastructure using Terraform and Kubernetes. Must have experience with CI/CD pipelines (Jenkins/GitHub Actions)."

# Match (DevOps)
res_devops_match = "Cloud Engineer with expertise in AWS, Terraform, and Kubernetes. Designed automated CI/CD pipelines using GitHub Actions and Docker."

# Mismatch (IT Support/System Admin)
res_devops_mismatch = "System Administrator with experience in Windows Server, Active Directory, and hardware troubleshooting. Managed local office networks and PC setups."

print(f"DevOps Match: {predict_score(res_devops_match, jd_devops):.4f}")
print(f"DevOps Mismatch: {predict_score(res_devops_mismatch, jd_devops):.4f}")

DevOps Match: 0.8194
DevOps Mismatch: 0.3299


In [20]:
jd_lead = "Engineering Manager to lead a team of 10 developers. Requires 3+ years of people management experience and Agile/Scrum certification."

# Match (Manager)
res_lead_match = "Engineering Manager with 4 years of experience leading cross-functional teams. Certified Scrum Master (CSM) with a focus on Agile delivery."

# Mismatch (Senior Individual Contributor)
res_lead_mismatch = "Principal Software Engineer with 15 years of coding experience in C++. Expert in kernel development and performance optimization. No management experience."

print(f"Lead Match: {predict_score(res_lead_match, jd_lead):.4f}")
print(f"Lead Mismatch: {predict_score(res_lead_mismatch, jd_lead):.4f}")

Lead Match: 0.7947
Lead Mismatch: 0.4383


In [22]:
res_text = """
Anurag Lawaniya
Software Engineering Intern | CITM College
Contact: anurag@example.com

PROFESSIONAL SUMMARY:
Highly motivated developer with a strong foundation in Python and JavaScript. 
Experienced in building scalable web applications and implementing machine 
learning models for real-world problems.

TECHNICAL SKILLS:
- Languages: Python, JavaScript (ES6+), SQL, HTML5, CSS3
- Frameworks/Libraries: Django, React.js, FastAPI, Scikit-learn, TensorFlow
- Tools: Docker, Git, AWS (S3, Lambda), PostgreSQL, MongoDB
- Concepts: RESTful APIs, NLP, Neural Networks, Microservices

PROJECTS:
1. Emotion Detection System: Developed an AI/ML application using NLP and 
   TensorFlow to analyze sentiment in real-time customer feedback.
2. Stock Prediction Portal: Built a full-stack dashboard using React and 
   Django to visualize financial trends.

EDUCATION:
Bachelor of Technology in Computer Science
CITM College | Expected 2025
"""

# --- Real World Job Description (Backend/Java Focus) ---
jd_text = """
Job Description

We are seeking a highly skilled and experienced Machine Learning and Data Engineer to join our dynamic team. The ideal candidate will be responsible for designing, developing, and maintaining data pipelines and machine learning models, ensuring optimal performance and scalability. They will work closely with data scientists, software engineers, and business stakeholders to implement data-driven solutions and deploy machine learning models in production environments.


Responsibilities

Design and develop scalable data pipelines and ETL processes to gather, process, and transform data from various sources.
Build and deploy machine learning models using state-of-the-art algorithms and techniques.
Maintain and optimize existing ML models to ensure performance and accuracy.
Collaborate with data scientists and software engineers to integrate models into production systems.
Conduct data preprocessing, feature engineering, and data wrangling for model training and evaluation.
Develop and maintain data architectures, including databases and data warehouses.
Implement data validation, data quality checks, and monitoring to ensure data integrity.
Automate model training, testing, and deployment processes.
Perform model evaluation and fine-tuning to enhance accuracy and efficiency.
Develop and maintain comprehensive documentation for models and data pipelines.
Stay updated with the latest trends and advancements in machine learning and data engineering.


Qualifications

Bachelor’s or master’s degree in Computer Science, Data Science, Engineering, or related field.
5+ years of experience in data engineering and machine learning.
Proficiency in programming languages like Python and Java.
Strong hands-on experience with machine learning frameworks such as TensorFlow, PyTorch, or Scikit-learn.
Expertise in data processing tools like Apache Spark, Hadoop, and SQL.
Familiarity with data visualization tools (e.g., Tableau, Power BI) and experience with libraries like Matplotlib.
Proficiency in database technologies (SQL and NoSQL).
Experience with cloud platforms (AWS, GCP, Azure) and containerization tools (Docker, Kubernetes).
Strong problem-solving and analytical skills with a keen eye for detail.
Excellent communication and collaboration abilities.
Certification in Machine Learning, Data Science, or related fields.
Experience with MLOps practices and tools (e.g., MLflow, Kubeflow).
Knowledge of big data processing and real-time data streaming (Kafka, Apache Flink).
Familiarity with data governance and data privacy standards.


Why Join Us?

Join a mission-driven team where your work directly will transform radiology through AI. mlHealth 360 is a global healthtech company transforming radiology with secure, cloud-native AI solutions. Our suite of medical imaging products improves diagnostic accuracy, reporting speed, and healthcare efficiency while seamlessly integrating into existing hospital systems. We offer a dynamic startup environment with opportunities for professional growth in the rapidly evolving field of digital health and AI regulation.


"""
print(f"Lead Match: {predict_score(res_text, jd_text):.4f}")

Lead Match: 0.7394
